In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import os
os.environ['CUDA_VISIBLE_DEVICES'] = '0'
os.environ["XLA_PYTHON_CLIENT_PREALLOCATE"]="false"
from functools import partial
import time
import glob
import pathlib
from tqdm import tqdm
import numpy as np
import matplotlib.pyplot as plt
from natsort import natsorted
import pandas as pd

import jax
import jax.numpy as jnp
import equinox as eqx


import matplotlib as mpl
from matplotlib import rc
mpl.rcParams['text.usetex'] = True
mpl.rcParams.update({'font.size': 10 * 2.54})
mpl.rcParams['text.latex.preamble']=r"\usepackage{bm}\usepackage{amsmath}"

In [ ]:
from dmpe.data_management import ClassicalSystems, DataPaths

In [ ]:
system_name = ClassicalSystems.CART_POLE
algo_name = "dmpe"

In [ ]:
result_path = DataPaths().sensitivity_analysis_experiments / pathlib.Path(
    f"sensitivity_analysis_data_{system_name.name.lower()}_{algo_name}.pkl"
)
results_df = pd.read_pickle(result_path)

In [ ]:
fig, axs = plt.subplots(5,1, figsize=(10, 8), sharex=True, constrained_layout=True)

for idx, metric_key in enumerate(["jsd", "ae", "mcudsa", "ksfc", "df"]):

    grouped = results_df.groupby("bandwidth")[metric_key].agg(["mean", "std"]).reset_index()

    axs[idx].plot(grouped["bandwidth"], grouped["mean"], marker="x")
    axs[idx].fill_between(
        grouped["bandwidth"],
        grouped["mean"] - grouped["std"],
        grouped["mean"] + grouped["std"],
        alpha=0.3,
    )
    axs[idx].set_ylabel(metric_key)

for ax in axs:
    ax.grid(alpha=0.5)
    ax.tick_params(which="both", axis="y", direction="in")
    ax.tick_params(which="both", axis="x", direction="in")
    ax.set_xscale("log")
    ax.set_yscale("log")

    
axs[-1].set_xlim(results_df["bandwidth"].min(), results_df["bandwidth"].max())
axs[-1].set_xlabel("$h$")

plt.show()